# RAGAS Evaluation   Tier 1

Runs five RAGAS metrics + Hit@k + MRR on the T1 batch evaluation output of
the Discriminative RAG pipeline. Uses Qwen as judge LLM via vLLM..

**Hardware:** Colab A100 (80 GB). 

**Inputs (upload to /content/):**
- `run_t1.xlsx`            pipeline output
- `all_questions.xlsx`     benchmark with ground truth + reference chunks
- `chunks_dota2.jsonl`     chunk_id → text for the Dota 2 corpus
- `chunks_lol.jsonl`       chunk_id → text for the League of Legends corpus

**Outputs:**
- `t1_ragas_per_question.xlsx` : one row per question, all metrics
- `t1_ragas_summary.xlsx`      : aggregate stats (overall + per-domain)

## 1. Install dependencies

In [ ]:
!pip install -q --upgrade vllm
!pip install -q ragas==0.2.10 datasets==3.0.0
!pip install -q langchain-openai==0.2.5 langchain-huggingface==0.1.0
!pip install -q sentence-transformers openpyxl pandas

## 2. Configuration

In [ ]:
import os
from pathlib import Path


JUDGE_MODEL = "Qwen/Qwen3.5-27B"
JUDGE_DISABLE_THINKING = True
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-4B"
EMBEDDING_DEVICE = "cpu"   # change to "cuda" if you have memory left
VLLM_HOST       = "127.0.0.1"
VLLM_PORT       = 8000
MAX_MODEL_LEN   = 8192          # well below Qwen3.5's 262K native; saves KV cache memory
GPU_UTIL        = 0.85          # 0.85 × 80 GB = 68 GB for vLLM, ~12 GB headroom for embedder
DTYPE           = "bfloat16"    # match the published Qwen3.5-27B weights
WORK_DIR        = Path("/content")
RUN_XLSX        = WORK_DIR / "run_t1.xlsx"
QUESTIONS_XLSX  = WORK_DIR / "all_questions.xlsx"
CHUNKS_JSONLS   = [
    WORK_DIR / "dota2_all_chunks.jsonl",
    WORK_DIR / "lol_all_chunks.jsonl",
]

OUT_PER_Q       = WORK_DIR / "t1_ragas_per_question.xlsx"
OUT_SUMMARY     = WORK_DIR / "t1_ragas_summary.xlsx"

TIER_FILTER     = "Tier-1"
JUDGE_TEMPERATURE = 0.0

## 3. Verify uploaded files

In [ ]:
required_files = [RUN_XLSX, QUESTIONS_XLSX, *CHUNKS_JSONLS]
for p in required_files:
    assert p.exists(), f"Missing: {p}"
    size_mb = p.stat().st_size / 1024 / 1024
    print(f"{p.name:30s}  {size_mb:7.2f} MB")

## 4. Build chunk_id

In [ ]:
import json

CHUNK_ID_FIELDS   = ["chunk_id", "id", "_id"]
CHUNK_TEXT_FIELDS = ["text", "content", "page_content", "chunk_text", "passage"]
METADATA_FIELDS   = ["metadata", "meta"]


def _first_present(rec: dict, candidates: list[str]):
    """Return the value of the first field in `candidates` that exists in `rec`."""
    for k in candidates:
        if k in rec:
            return rec[k]
    return None


def _construct_chunk_id(rec: dict):
    meta = rec.get("metadata") or rec.get("meta") or {}
    if not isinstance(meta, dict):
        return None
    game = meta.get("game")
    if game is None:
        return None
    entity = (
        meta.get("hero")
        or meta.get("champion")
        or meta.get("item")
        or meta.get("mechanics")
        or "general"
    )
    full_path = meta.get("full_path", "unknown_path")
    chunk_index = meta.get("chunk_index", 0)
    return f"{game}_{entity}_{full_path}_{chunk_index}"


def _resolve_chunk_id(rec: dict) -> tuple:
    """Find chunk_id: top-level → metadata → construct from metadata."""
    for k in CHUNK_ID_FIELDS:
        if k in rec:
            return rec[k], k
    for meta_key in METADATA_FIELDS:
        meta = rec.get(meta_key)
        if isinstance(meta, dict):
            for k in CHUNK_ID_FIELDS:
                if k in meta:
                    return meta[k], f"{meta_key}.{k}"
    constructed = _construct_chunk_id(rec)
    if constructed is not None:
        return constructed, "constructed from metadata"
    return None, None


def load_chunks_jsonl(path: Path) -> dict[str, str]:
    """Read {chunk_id: text} from a JSONL with flexible field detection."""
    out: dict[str, str] = {}
    id_loc, text_field = None, None
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            cid, this_id_loc = _resolve_chunk_id(rec)
            text = _first_present(rec, CHUNK_TEXT_FIELDS)

            if line_no == 1:
                id_loc = this_id_loc
                text_field = next((k for k in CHUNK_TEXT_FIELDS if k in rec), None)
                if cid is None or text is None:
                    meta_keys = (
                        list(rec.get("metadata", {}).keys())
                        if isinstance(rec.get("metadata"), dict) else []
                    )
                    raise ValueError(
                        f"{path.name}: could not detect/construct id or detect text.\n"
                        f"  Top-level keys: {list(rec.keys())}\n"
                        f"  metadata keys:  {meta_keys}"
                    )

            if cid is None or text is None:
                continue
            out[str(cid)] = text
    print(f"  {path.name}: {len(out):>5} chunks  (id: {id_loc}, text: '{text_field}')")
    # Sanity-check sample for join compatibility with all_questions.xlsx
    for sid in list(out.keys())[:2]:
        print(f"    sample: {sid}")
    return out


chunk_text_by_id: dict[str, str] = {}
for jsonl_path in CHUNKS_JSONLS:
    chunks = load_chunks_jsonl(jsonl_path)
    overlap = set(chunks) & set(chunk_text_by_id)
    if overlap:
        print(f"{len(overlap)} duplicate chunk_ids - keeping first-seen value")
        chunks = {k: v for k, v in chunks.items() if k not in chunk_text_by_id}
    chunk_text_by_id.update(chunks)

print(f"\nTotal: {len(chunk_text_by_id):,} chunks")

## 5. Start vLLM server in background


In [ ]:
import time
import urllib.request
import urllib.error

# Kill any prior server (e.g. from a previous failed run)
subprocess.run(["pkill", "-f", "vllm.entrypoints.openai.api_server"], check=False)
time.sleep(2)

vllm_log_path = "/content/vllm.log"
vllm_log_f = open(vllm_log_path, "w")
vllm_proc = subprocess.Popen(
    [
        "python", "-m", "vllm.entrypoints.openai.api_server",
        "--model", JUDGE_MODEL,
        "--host", VLLM_HOST,
        "--port", str(VLLM_PORT),
        "--max-model-len", str(MAX_MODEL_LEN),
        "--gpu-memory-utilization", str(GPU_UTIL),
        "--dtype", DTYPE,
        "--no-enable-log-requests",
    ],
    stdout=vllm_log_f, stderr=subprocess.STDOUT,
)
print(f"vLLM PID {vllm_proc.pid} → {vllm_log_path}")

## 6. Wait for vLLM to be ready

In [ ]:
HEALTH_URL = f"http://{VLLM_HOST}:{VLLM_PORT}/v1/models"
TIMEOUT_S  = 2400 


def _dump_log_tail(n: int = 40) -> None:
    print(f"\nlast {n} lines of {vllm_log_path}")
    try:
        with open(vllm_log_path) as f:
            for line in f.readlines()[-n:]:
                print(line.rstrip())
    except Exception as e:
        print(f"(could not read log: {e})")


t_start = time.time()
last_progress = t_start
while True:
    if vllm_proc.poll() is not None:
        _dump_log_tail()
        raise RuntimeError("vLLM process died, see log dump above")
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=2) as r:
            if r.status == 200:
                print(f"\nvLLM ready after {int(time.time() - t_start)}s")
                break
    except (urllib.error.URLError, ConnectionError):
        pass
    elapsed = time.time() - t_start
    if elapsed > TIMEOUT_S:
        _dump_log_tail()
        raise TimeoutError(f"vLLM not ready after {TIMEOUT_S}s")
    # Heartbeat every 30s so you know it's alive
    if time.time() - last_progress > 30:
        print(f"  ...still waiting ({int(elapsed)}s elapsed)")
        last_progress = time.time()
    time.sleep(5)

## 7. Sanity-check the endpoint

Two checks: (a) the endpoint responds, (b) thinking mode is actually
disabled: if `<think>` leaks through, RAGAS rubric outputs will fail
to parse downstream.

In [ ]:
from openai import OpenAI

client = OpenAI(base_url=f"http://{VLLM_HOST}:{VLLM_PORT}/v1", api_key="dummy")

# Build the extra_body once so we can reuse the exact same kwargs in RAGAS
JUDGE_EXTRA_BODY: dict = {}
if JUDGE_DISABLE_THINKING:
    JUDGE_EXTRA_BODY["chat_template_kwargs"] = {"enable_thinking": False}

ping = client.chat.completions.create(
    model=JUDGE_MODEL,
    messages=[{"role": "user", "content": "Reply with exactly: ready"}],
    temperature=0.0,
    max_tokens=20,
    extra_body=JUDGE_EXTRA_BODY,
)
ping_text = ping.choices[0].message.content.strip()
print(f"Judge says: {ping_text!r}")

if "<think>" in ping_text or "</think>" in ping_text:
    raise RuntimeError(
        "Thinking-mode leak detected   RAGAS outputs will be polluted.\n"
        "  Either set JUDGE_DISABLE_THINKING = False (if you don't care, "
        "and add a downstream <think> stripper), or check your vLLM version "
        "supports the `chat_template_kwargs` extra_body parameter."
    )
print("thinking mode confirmed off")

## 8. Wire RAGAS to the local judge + embeddings

In [ ]:
!pip uninstall -y torchcodec

In [ ]:
# Workaround for langchain-core <-> langchain version mismatch in Colab
import langchain
for attr, default in [("verbose", False), ("debug", False), ("llm_cache", None)]:
    if not hasattr(langchain, attr):
        setattr(langchain, attr, default)

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

# Pass extra_body as a direct parameter (newer langchain-openai API)
judge_lc = ChatOpenAI(
    model=JUDGE_MODEL,
    base_url=f"http://{VLLM_HOST}:{VLLM_PORT}/v1",
    api_key="dummy",
    temperature=JUDGE_TEMPERATURE,
    max_tokens=1024,
    extra_body=JUDGE_EXTRA_BODY or None,
)
judge_llm = LangchainLLMWrapper(judge_lc)

embeddings_lc = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": EMBEDDING_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)
embeddings = LangchainEmbeddingsWrapper(embeddings_lc)
print(f"RAGAS LLM + embeddings configured (embeddings on {EMBEDDING_DEVICE})")
if JUDGE_EXTRA_BODY:
    print(f"  judge extra_body: {JUDGE_EXTRA_BODY}")

## 9. Load and join data

In [ ]:
import pandas as pd

run_df = pd.read_excel(RUN_XLSX)
qs_df  = pd.read_excel(QUESTIONS_XLSX)

# Normalize both sides to lowercase 'question' so we have a single join key
def _normalize_question_col(df, name):
    for cand in ["question", "Query", "query", "Question"]:
        if cand in df.columns:
            if cand != "question":
                df = df.rename(columns={cand: "question"})
                print(f"  {name}: renamed '{cand}' -> 'question'")
            return df
    raise ValueError(f"{name}: no question/query column found. Columns: {list(df.columns)}")

run_df = _normalize_question_col(run_df, "run_df")
qs_df  = _normalize_question_col(qs_df, "qs_df")

if "Tier" in run_df.columns:
    run_df = run_df.rename(columns={"Tier": "predicted_tier"})

run_query_col = "question"
join_key      = "question"

merged = run_df.merge(qs_df, on=join_key, how="inner")

merged = merged[merged["Tier"] == TIER_FILTER].reset_index(drop=True)

print(f"Joined on '{join_key}'   {len(merged)} {TIER_FILTER} rows")
print(f"Run had {(run_df['error'].notna() & (run_df['error'] != '')).sum() if 'error' in run_df.columns else 0} pipeline errors (excluded if any)")

## 11. Build RAGAS dataset

Each row needs four fields:
- `user_input`         : the original question
- `response`           : the generated answer
- `retrieved_contexts` : list of chunk *texts* in rank order (score DESC)
- `reference`          : ground-truth answer (for Context Recall, Correctness)

In [ ]:
PRETTY_TO_SNAKE = {
    "Answer":                          "answer",
    "Admitted Chunk IDs (Ranked)":     "admitted_chunk_ids_ranked",
    "Gated-Out Chunk IDs (Ranked)":    "gated_out_chunk_ids_ranked",
    "Source Domains":                  "source_domains",
    "Error":                           "error",
    "Expected Domain":                 "expected_domain",
    "Expected Decision":               "expected_decision",
}

renamed = {k: v for k, v in PRETTY_TO_SNAKE.items() if k in merged.columns}
merged = merged.rename(columns=renamed)
print(f"Renamed {len(renamed)} columns: {list(renamed.values())}")

# Sanity-check the same row that came up empty before
row = merged.iloc[0]
print(f"\n  question:      {str(row['question'])[:80]!r}")
print(f"  answer:        {str(row.get('answer', 'STILL MISSING'))[:80]!r}")
print(f"  ground_truth:  {str(row.get('ground_truth'))[:80]!r}")
chunk_ids_str = row.get("admitted_chunk_ids_ranked", "")
print(f"  chunk_ids raw: {str(chunk_ids_str)[:120]!r}")

if chunk_ids_str:
    first_id = str(chunk_ids_str).split("|")[0].strip()
    print(f"  first chunk_id: {first_id!r} → corpus hit: {first_id in chunk_text_by_id}")

In [ ]:
def parse_pipe_list(s) -> list[str]:
    if pd.isna(s) or not s:
        return []
    return [tok.strip() for tok in str(s).split("|") if tok.strip()]


def chunk_ids_to_texts(ids: list[str]) -> list[str]:
    """Look up text for each chunk_id; warn on misses."""
    texts = []
    for cid in ids:
        text = chunk_text_by_id.get(cid)
        if text is None:
            print(f"missing chunk in corpus: {cid}")
            continue
        texts.append(text)
    return texts


samples = []
n_skipped = 0
for _, row in merged.iterrows():
    answer = row.get("answer")
    if not isinstance(answer, str) or not answer.strip():
        n_skipped += 1
        continue

    chunk_ids = parse_pipe_list(row.get("admitted_chunk_ids_ranked"))
    contexts = chunk_ids_to_texts(chunk_ids)
    if not contexts:
        n_skipped += 1
        continue

    samples.append({
        "user_input":         str(row[run_query_col]),
        "response":           str(answer),
        "retrieved_contexts": contexts,
        "reference":          str(row["ground_truth"]),
    })

print(f"Built {len(samples)} samples ({n_skipped} skipped: empty answer or no admitted chunks)")

In [ ]:
from ragas import EvaluationDataset

eval_dataset = EvaluationDataset.from_list(samples)
print(f"EvaluationDataset: {len(eval_dataset)} samples")

## 12. Run RAGAS metrics

In [ ]:
!pip uninstall -y torchcodec -q

# Clear partial imports so sentence-transformers re-imports cleanly
import sys
for mod_name in list(sys.modules.keys()):
    if mod_name.startswith(("sentence_transformers", "torchcodec")):
        del sys.modules[mod_name]

from langchain_huggingface import HuggingFaceEmbeddings
from ragas.embeddings import LangchainEmbeddingsWrapper

embeddings_lc = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": EMBEDDING_DEVICE},
    encode_kwargs={"normalize_embeddings": True},
)
embeddings = LangchainEmbeddingsWrapper(embeddings_lc)
print(f"embeddings ready on {EMBEDDING_DEVICE}")

In [ ]:
from ragas import evaluate
from ragas.metrics import (
    Faithfulness,
    LLMContextRecall,
    LLMContextPrecisionWithReference,
    AnswerCorrectness,
    AnswerRelevancy,
)

metrics = [
    Faithfulness(),
    LLMContextRecall(),
    LLMContextPrecisionWithReference(),
    AnswerCorrectness(),
    AnswerRelevancy(),
]

result = evaluate(
    dataset=eval_dataset,
    metrics=metrics,
    llm=judge_llm,
    embeddings=embeddings,
    show_progress=True,
    raise_exceptions=False,   # let per-row failures be NaN, don't crash
)

ragas_df = result.to_pandas()
print(ragas_df.describe())

## 13. Local IR metrics: Hit@k and MRR

Pure set/rank operations on chunk IDs. T1 questions can have 1–3 ground-truth
chunks (across `source_chunk_id_1..6`). A retrieval is credited if **any**
GT chunk appears.

In [ ]:
SOURCE_CHUNK_COLS = [f"source_chunk_id_{i}" for i in range(1, 7)]


def gt_chunk_set(row) -> set[str]:
    out = set()
    for col in SOURCE_CHUNK_COLS:
        v = row.get(col)
        if isinstance(v, str) and v.strip():
            out.add(v.strip())
    return out


def hit_at_k(retrieved: list[str], gt: set[str], k: int) -> int:
    return int(any(cid in gt for cid in retrieved[:k]))


def reciprocal_rank(retrieved: list[str], gt: set[str]) -> float:
    for rank, cid in enumerate(retrieved, start=1):
        if cid in gt:
            return 1.0 / rank
    return 0.0


def full_ranked(row) -> list[str]:
    admitted = parse_pipe_list(row.get("admitted_chunk_ids_ranked"))
    gated_out = parse_pipe_list(row.get("gated_out_chunk_ids_ranked"))
    return admitted + gated_out  # both already score-DESC sorted


ir_rows = []
for _, row in merged.iterrows():
    ranked = full_ranked(row)
    gt = gt_chunk_set(row)
    if not gt:
        ir_rows.append({"hit@1": None, "hit@3": None, "hit@5": None, "hit@10": None, "mrr": None})
        continue
    ir_rows.append({
        "hit@1":  hit_at_k(ranked, gt, 1),
        "hit@3":  hit_at_k(ranked, gt, 3),
        "hit@5":  hit_at_k(ranked, gt, 5),
        "hit@10": hit_at_k(ranked, gt, 10),
        "mrr":    reciprocal_rank(ranked, gt),
    })

ir_df = pd.DataFrame(ir_rows)
print(ir_df.describe())

## 14. Combine and save

In [ ]:
def is_evaluable(row) -> bool:
    answer = row.get("answer")
    if not isinstance(answer, str) or not answer.strip():
        return False
    if not parse_pipe_list(row.get("admitted_chunk_ids_ranked")):
        return False
    return True


merged_eval = merged[merged.apply(is_evaluable, axis=1)].reset_index(drop=True)
ir_df_eval  = ir_df[merged.apply(is_evaluable, axis=1).values].reset_index(drop=True)

assert len(merged_eval) == len(ragas_df), \
    f"length mismatch: merged_eval={len(merged_eval)} ragas_df={len(ragas_df)}"

# Per-question output
keep_cols = [
    "qid" if "qid" in merged_eval.columns else run_query_col,
    run_query_col, "Tier", "ground_truth", "answer",
    "expected_domain" if "expected_domain" in merged_eval.columns else None,
]
keep_cols = [c for c in keep_cols if c is not None and c in merged_eval.columns]

per_q = pd.concat(
    [
        merged_eval[keep_cols].reset_index(drop=True),
        ragas_df.reset_index(drop=True),
        ir_df_eval.reset_index(drop=True),
    ],
    axis=1,
)
per_q.to_excel(OUT_PER_Q, index=False)
print(f"{OUT_PER_Q}  ({len(per_q)} rows)")

# Summary stats
metric_cols = [
    "faithfulness",
    "context_recall",
    "llm_context_precision_with_reference",
    "answer_correctness",
    "answer_relevancy",
    "hit@1", "hit@3", "hit@5", "hit@10", "mrr",
]
present_metrics = [c for c in metric_cols if c in per_q.columns]

summary_rows = [
    {"slice": "T1-overall", "n": len(per_q), **{m: per_q[m].mean() for m in present_metrics}},
]
if "expected_domain" in per_q.columns:
    for dom in sorted(per_q["expected_domain"].dropna().unique()):
        sub = per_q[per_q["expected_domain"] == dom]
        summary_rows.append({
            "slice": f"T1 - {dom}",
            "n": len(sub),
            **{m: sub[m].mean() for m in present_metrics},
        })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_excel(OUT_SUMMARY, index=False)
print(f"{OUT_SUMMARY}")
print()
print(summary_df.to_string(index=False))

## 15. Download outputs

In [ ]:
from google.colab import files  # type: ignore
files.download(str(OUT_PER_Q))
files.download(str(OUT_SUMMARY))

## 16. Cleanup

Kills the vLLM process so the GPU is released for the next session.

In [ ]:
vllm_proc.terminate()
try:
    vllm_proc.wait(timeout=10)
except subprocess.TimeoutExpired:
    vllm_proc.kill()
print("vLLM stopped")